# ICR이 유난히 크거나 작은 여섯 관측치가 결론을 바꿀까?

ICR은 분수다. 영업이익이 같아도 이자비용이 아주 작으면 값이 크게 뛴다. 전체표본을 기준으로 삼고 양쪽 끝의 관측치를 잠시 빼서 비교했다.

ICR(이자보상배율)은 영업이익을 이자비용으로 나눈 값이다. 3배라면 영업이익이 그해 이자비용의 세 배라는 뜻이다. 한 행은 한 업종의 한 해다. 코드의 E는 사전 단기차입 의존도, L은 사전 유동비율이다. Post는 2022~2023년이며 EP는 E와 Post를 곱한 항이다. EP의 계수는 사전 단기차입 의존도가 1%포인트 높은 업종에서 긴축기의 상대적 ICR 변화가 몇 배 달랐는지 나타낸다.


[전체 질문 목록](../README.md)

**표 읽는 법**　N은 업종·연도 관측 수, G는 업종 수다. 회귀표의 beta는 추정한 관계의 크기, lo와 hi는 95% 신뢰구간의 양 끝이다. p값은 계수가 0이라는 가정에서 지금만큼 또는 더 극단적인 검정통계량이 나올 확률이다.

# ICR 극단 관측치 제외 민감도 분석
사용자 요청(2026-11-02): 기존 기간모형 M1–M4 및 M2 기반 M5에서 극단값 제외가 타당한지 검토하고 실행. 원 분석을 보존한 보조 분석이며 데이터 오류 판정·주 모형 교체가 아니다.

## 고정한 사양
- 2018–2023년 40개 업종, Post=2022–23, D2021 별도. 사전 E,L=기존 2018–20 평균을 유지. E=100×(단기차입+단기사채)/자산; L=100×유동자산/유동부채의 연도별 비율 평균. Y=ICR(배).
- 전체 240개 관측치 Y의 선형보간 1%,99% 분위수를 한 번 계산하고 구간 밖 행만 제외. 업종 전체를 지우지 않음. 설명변수·원자재·환율은 제외 기준으로 사용하지 않음. 음수 ICR도 무조건 제외하지 않음.
- 1%는 공식적 이상치 검정 임계치가 아닌 분석자 선택. 기존 M2의 1/99% 윈저화와 같은 경계를 사용해 행 삭제와 값 제한을 구분한다. 결과에 따라 경계를 조정하지 않는다. 연도별 분위수나 5% 절단 등 추가 사양을 탐색하지 않는다.
- 모든 모형 업종·연도 고정효과. M1=EP+ET, M2=EP+LP+ET+LT, M3=M2+shock, M4=M3+FX, M5=M2+ELP+ELT(중심화 E,L). M5 중심은 원 표본 40업종 평균에 고정.
- shock 및 FX는 기존 최종 병합패널 열을 변경 없이 이용. 원자재 충격과 수출유발 비중×환율 로그변화의 기존 산식·단위·2019 가중치는 이전 제출노트북 참조. 이번 작업은 입력 패널 이후의 민감도 분석을 독립 재현한다.
- CR1 업종 군집표준오차, t(G−1) 95% CI. M2 EP/LP 및 M5 ELP는 WCR-S(31)·WCR-C(11) 9,999회, seed=20261102. 다중비교 미조정, 절단 경계·표본 선택을 고정한 조건부 추론.

## 근거와 한계
[NIST/SEMATECH e-Handbook §1.3.5.17](https://itl.nist.gov/div898/handbook/eda/section3/eda35h.htm), Introduction의 관련 본문 확인: “we typically do not want to simply delete the outlying observation”. 오류가 확인되지 않은 극단값을 자동 삭제하지 말라는 근거이며, 1% 절단을 권고하는 자료가 아니다. 본 분석은 원 표본 결과와의 민감도 비교다. Y로 표본을 제한하므로 원래 전체 업종·기간의 관계와 달라질 수 있고, 오류 교정이나 더 정확한 추정으로 단정할 수 없다. 극단값이 실제 취약 업종의 신호일 가능성도 있다.

재실행: Python 3.12, requirements.txt 환경에서 이 폴더를 작업경로로 전체 셀 실행. inputs 파일의 해시 확인부터 실행한다. 모든 CSV·입력·노트북을 함께 보관한다.


In [1]:
# 전체 표본 확인
from pathlib import Path
import json,hashlib
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from IPython.display import display
from wildboottest.wildboottest import wildboottest
BASE = Path.cwd()
(BASE / 'tables').mkdir(exist_ok=True)
def save(x, n):
    x.to_csv(BASE / 'tables' / f'{n}.csv', index=False, encoding='utf-8-sig')
for v in json.loads((BASE/'inputs/manifest.json').read_text(encoding='utf-8')):
 assert hashlib.sha256((BASE/'inputs'/v['file']).read_bytes()).hexdigest()==v['sha256']
t = pd.read_csv(BASE / 'inputs/panel.csv')
assert len(t) == 240 and t.code.nunique() == 40
assert not t.duplicated(['code','year']).any()
assert np.isfinite(t[['Y','EP','ET','LP','LT','shock','FX']]).all().all()

# 양끝에 있는 관측 추출
low,high=t.Y.quantile([.01,.99],interpolation='linear')
removed = t.loc[~t.Y.between(low, high)].copy()
trim = t.loc[t.Y.between(low, high)].copy().reset_index(drop=True)
removed['reason']=np.where(removed.Y<low,'ICR < pooled P1','ICR > pooled P99')
save(removed, 'excluded_observations')
save(trim, 'trimmed_panel')
save(pd.DataFrame([{'lower':low,'upper':high,'N_original':len(t),'N_trim':len(trim),'G_original':t.code.nunique(),'G_trim':trim.code.nunique()}]),'sample_audit')
save(pd.concat([t.groupby('year').size().rename('original'),trim.groupby('year').size().rename('trimmed')],axis=1).reset_index(),'sample_by_year')
print('ICR boundaries', low, high)
display(removed[['code', 'industry', 'year', 'Y', 'reason']])

# 유동성 결합항 정의
centers=t.drop_duplicates('code')[['E','L']].mean()
def moderation(d):
    c = d.copy()
    e = c.E - centers.E
    l = c.L - centers.L
    c['EP'] = e * c.Post
    c['LP'] = l * c.Post
    c['ET'] = e * (c.year == 2021)
    c['LT'] = l * (c.year == 2021)
    c['ELP'] = e * l * c.Post
    c['ELT'] = e * l * (c.year == 2021)
    return c


ICR boundaries -7.2226300000000005 82.87627999999987
      code industry  year        Y            reason
19   I3104      컴퓨터  2019   -9.300   ICR < pooled P1
29   I3105     통신기기  2023   -9.630   ICR < pooled P1
177  I3312       조선  2021   -7.372   ICR < pooled P1
188  I3402       담배  2020   86.444  ICR > pooled P99
189  I3402       담배  2021  129.613  ICR > pooled P99
190  I3402       담배  2022  134.803  ICR > pooled P99


### 출력에서 읽을 수 있는 것

하단 경계보다 낮은 세 행은 컴퓨터 2019년 −9.300배, 통신기기 2023년 −9.630배, 조선 2021년 −7.372배다. 상단 세 행은 담배의 2020~2022년이다.

- 전체 표본은 240행, 40개 업종이며 주요 변수의 유한값과 관측키를 점검했다.
- 양끝 1% 밖의 6행은 컴퓨터 2019, 통신기기 2023, 조선 2021, 담배 2020~2022였다. 특정 업종을 통째로 제외한 것이 아니므로 40개 업종은 모두 남는다.
- M5에서 사용하는 결합항을 만들 때 전체표본에서 잡은 중심값을 유지하도록 함수를 정의했다.

In [2]:
# 공통 추정식 준비
def design(d,terms):
    X=pd.concat([pd.DataFrame({'const':np.ones(len(d))},index=d.index),d[terms],pd.get_dummies(d.code,prefix='i',drop_first=True,dtype=float),pd.get_dummies(d.year,prefix='y',drop_first=True,dtype=float)],axis=1).astype(float)
    assert np.linalg.matrix_rank(X)==X.shape[1]
    return X
fits = {}
rows = []
def fit(label, d, terms, y='Y'):
    X = design(d, terms)
    model = sm.OLS(d[y].astype(float), X)
    res = model.fit(cov_type='cluster', cov_kwds={'groups': d.code, 'use_correction': True, 'df_correction': True}, use_t=True)
    G = d.code.nunique()
    q = stats.t.ppf(0.975, G - 1)
    for term in terms:
        b = float(res.params[term])
        se = float(res.bse[term])
        rows.append(dict(model=label, outcome=y, term=term, beta=b, se=se, lo=b - q * se, hi=b + q * se, p=float(2 * stats.t.sf(abs(b / se), G - 1)), N=len(d), G=G))
    fits[label] = (d.copy(), terms, model, res)
    return res

# 전체와 제한표본 추정
m2=['EP','LP','ET','LT']
specs={'M1':['EP','ET'],'M2':m2,'M3':m2+['shock'],'M4':m2+['shock','FX'],'M5':m2+['ELP','ELT']}
for sample, d in [('full', t), ('trim', trim)]:
    for name, terms in specs.items():
        fit(sample + '_' + name, moderation(d) if name == 'M5' else d, terms)
results = pd.DataFrame(rows)
save(results, 'all_coefficients')
oldmain = pd.read_csv(BASE / 'inputs/original_main.csv')
oldm5 = pd.read_csv(BASE / 'inputs/original_m5.csv')
for name in specs:
 expected=(oldm5[oldm5.model=='M5_exploratory'] if name=='M5' else oldmain[oldmain.model==name]).set_index('term')
 actual=results[results.model=='full_'+name].set_index('term')
 np.testing.assert_allclose(actual.loc[expected.index,'beta'],expected.beta,atol=1e-8)
focus=results[((results.model.str.endswith('M5')) & (results.term=='ELP'))|((~results.model.str.endswith('M5')) & (results.term=='EP'))]
save(focus, 'comparison')
display(focus)

# 제한표본 표준오차 검산
# 불균형 표본에서 업종 군집 공분산을 독립 행렬 계산으로 확인
dd, _, model, res = fits['trim_M2']
X = model.exog
y = model.endog
b = np.linalg.lstsq(X, y, rcond=None)[0]
u = y - X @ b
bread = np.linalg.inv(X.T @ X)
score = np.array([X[dd.code == g].T @ u[dd.code == g] for g in sorted(dd.code.unique())])
G = dd.code.nunique()
N = len(dd)
cov=bread@score.T@score@bread*G/(G-1)*(N-1)/(N-X.shape[1])
np.testing.assert_allclose(cov,res.cov_params(),atol=1e-7)
print('All original M1–M5 reproduced; trimmed M2 CR1 verified')


      model outcome term      beta  ...        hi         p    N   G
0   full_M1       Y   EP  0.299918  ...  1.201058  0.504794  240  40
2   full_M2       Y   EP  0.545912  ...  1.459896  0.234272  240  40
6   full_M3       Y   EP  0.570552  ...  1.435228  0.189726  240  40
11  full_M4       Y   EP  0.516299  ...  1.299743  0.190274  240  40
21  full_M5       Y  ELP -0.002557  ...  0.027943  0.866202  240  40
23  trim_M1       Y   EP  0.306599  ...  0.977714  0.361135  234  40
25  trim_M2       Y   EP  0.350346  ...  1.171716  0.393545  234  40
29  trim_M3       Y   EP  0.396856  ...  1.172670  0.307193  234  40
34  trim_M4       Y   EP  0.355313  ...  1.039166  0.299755  234  40
44  trim_M5       Y  ELP  0.006059  ...  0.021273  0.425358  234  40

[10 rows x 10 columns]
All original M1–M5 reproduced; trimmed M2 CR1 verified


### 출력에서 읽을 수 있는 것

full_M2와 trim_M2의 EP 행을 비교하면 관측 수는 240→234이고 업종 수는 40으로 같다. 제한표본의 95% 구간은 −0.471~+1.172배다.

- 전체표본과 제한표본에 같은 회귀식을 적용할 준비를 했다.
- M2에서 6행을 제외하면 +0.350배(p=0.394), M5 결합항은 +0.0061(p=0.425)이었다. 극단값을 제외해도 초기 가설이 뚜렷해지지 않았다.
- 업종별 남은 연도 수가 다른 제한표본에서도 표준오차 계산이 독립 계산과 일치했다.

In [3]:
# 업종 단위 재검정
B = 9999
SEED = 20261102
def within_boot(model, groups, param, label):
    xx = pd.DataFrame(model.exog, columns=model.exog_names)
    yy = pd.Series(model.endog)
    gg = pd.Series(np.asarray(groups))
    xx = xx - xx.groupby(gg).transform('mean')
    yy = yy - yy.groupby(gg).transform('mean')
    xx = xx.loc[:, np.max(np.abs(xx), axis=0) > 1e-10]
    assert np.linalg.matrix_rank(xx) == xx.shape[1]
    mm = sm.OLS(yy, xx)
    np.testing.assert_allclose(mm.fit().params[param], model.fit().params[param], atol=1e-08)
    ans = []
    for variant in ['31', '11']:
        res = wildboottest(mm, param=param, cluster=pd.factorize(gg)[0], B=B, seed=SEED, bootstrap_type=variant, impose_null=True, weights_type='rademacher', show=False)
        ans.append({'model': label, 'term': param, 'bootstrap_type': variant, 'p_boot': float(res['p-value'].iloc[0]), 'B': B, 'seed': SEED})
    return ans

boot=[]
for label, term in [('trim_M2', 'EP'), ('trim_M2', 'LP'), ('trim_M5', 'ELP')]:
    dd, _, model, res = fits[label]
    boot.extend(within_boot(model, dd.code, term, label))
boot = pd.DataFrame(boot)
save(boot, 'bootstrap')
display(boot)


     model term bootstrap_type    p_boot     B      seed
0  trim_M2   EP             31  0.506251  9999  20261102
1  trim_M2   EP             11  0.508451  9999  20261102
2  trim_M2   LP             31  0.646565  9999  20261102
3  trim_M2   LP             11  0.646565  9999  20261102
4  trim_M5  ELP             31  0.412141  9999  20261102
5  trim_M5  ELP             11  0.410841  9999  20261102


### 출력에서 읽을 수 있는 것

제한표본에서 M2의 단기차입 항 p_boot=0.506, 유동성 항은 0.647, M5 결합항은 0.412였다.

# 극단값 제외 분석 결과

결론: 산업 단위 M1–M4 및 M2 기반 M5를 유지해 실행했다. ICR 양끝 1% 밖 관측치를 제외해도 사전 단기차입 취약성 및 유동성 완충 가설을 뒷받침하는 통계적 근거가 확인되지 않았다. 이번 제외 기준에서는 극단 관측치가 기존 비유의 결과를 설명한다는 증거를 얻지 못했다. 모든 종류의 영향점 문제가 없다는 결론은 아니다.

## 변경 내용과 범위
기존 전체 240행 → ICR의 전체기간 1%/99% 분위수 [-7.222630, 82.876280] 밖 6행 제외 → 234행, 40업종 유지. 6/240=2.5%이며 유한 표본의 분위수 보간 때문에 정확히 2%가 아니다. 사전 E/L, Post=2022–23, 별도 2021 항, 통제변수와 M5 중심화 기준은 원본과 동일하다. 실제 금리 연속모형으로 바꾸지 않았다. 업종 전체 삭제도 아니다. 이번 실행은 사용자 요청에 따른 별도 보조 분석이며 원자료·기존 주 분석을 수정하지 않았다.

1%는 오류 판정의 공식 기준이 아니라 기존 윈저화와 비교하기 위한 분석자 선택이다. 기존 윈저화는 행을 유지하면서 경계값으로 제한했고, 이번 절단은 해당 행을 제외했다. 원래 표본의 노출도와 M5 중심화 평균을 유지해 표본 제외 이외의 변경을 막았다. 이 관측치들이 잘못된 값이라는 증거는 이번 작업에서 확인하지 않았다.

## 주요 결과
아래 p값은 업종 군집 CR1 및 t(39) 기준이며, M1–M4는 단기차입×Post, M5는 단기차입×유동비율×Post 결합항이다. 모형 전체의 유의성을 검정한 표가 아니다.

| 모형 | 원본 계수 | 제외 후 계수 | 제외 후 95% CI | 제외 후 p |
|---|---:|---:|---|---:|
| M1 | 0.2999 | 0.3066 | [-0.3645, 0.9777] | 0.3611 |
| M2 (주 모형) | 0.5459 | 0.3503 | [-0.4710, 1.1717] | 0.3935 |
| M3 | 0.5706 | 0.3969 | [-0.3790, 1.1727] | 0.3072 |
| M4 | 0.5163 | 0.3553 | [-0.3285, 1.0392] | 0.2998 |
| M5 결합항 | -0.00256 | 0.00606 | [-0.00915, 0.02127] | 0.4254 |

M2의 유동비율×Post는 +0.01261, 95% CI [-0.04714, 0.07235], p=0.6719다. M2 단기차입·유동비율 및 M5 결합항의 WCR-S p값은 각각 0.5063, 0.6466, 0.4121이다(9,999회, seed=20261102). 군집 부트스트랩을 사용해도 핵심 해석은 같다. 5%는 관례이지 진실 판정선이 아니며, p값·신뢰구간을 함께 공개한다.

M2의 계수는 사전 단기차입 비중 1%p 차이에 따른 2022–23 대 2018–20 상대 ICR 변화 차이(배)다. 제외 후에는 업종별 남은 연도가 달라져 균형패널의 단순 평균 대비와 완전히 같지는 않다. 금리의 전체 인과효과나 관계가 정확히 0이라는 증거로 해석하지 않는다.

## 부차적 결과도 함께 공개
M3 원자재 충격 계수는 원본 +0.1305(p=0.5022)에서 +0.2541(p=0.0026), M4에서는 +0.1341(p=0.4942)에서 +0.2536(p=0.0036)으로 달라졌다. M4 환율 항은 -0.7706(p=0.1456)이다. 따라서 모든 계수가 극단값 처리에 둔감하다고 말할 수 없다. 원자재 결과는 표본 선택에 민감한 보조 결과이며, 단기차입 가설의 입증이나 원자재 가격 상승의 유익한 인과효과로 바꾸어 해석하지 않는다. 원자재 항의 별도 부트스트랩은 이번 사전 지정 검산 범위에 포함하지 않았다.

## 제외한 관측치
- 컴퓨터 2019: ICR -9.300배
- 통신기기 2023: -9.630배
- 조선 2021: -7.372배
- 담배 2020: 86.444배
- 담배 2021: 129.613배
- 담배 2022: 134.803배

실제 적자나 작은 이자비용 때문에 발생한 값일 수도 있다. 제외 후 결과를 더 정확한 결과라고 단정하지 않으며 원본을 주 분석으로 유지하는 것을 권고한다. 사후 결과변수 Y로 표본을 제한하므로 취약성 연구에서 중요한 관측치까지 빠질 수 있다. 부트스트랩 p값도 제외 경계와 표본을 고정한 조건부 값이다.

## 근거와 다음 판단
[NIST/SEMATECH e-Handbook §1.3.5.17](https://itl.nist.gov/div898/handbook/eda/section3/eda35h.htm), Introduction 관련 본문을 확인했다. 인용: “we typically do not want to simply delete the outlying observation”. 오류 확인 없는 자동 삭제를 경계하는 근거로 사용한다. 이 문헌이 1% 제외를 공식 기준으로 제시한 것은 아니다.

지금은 제외 폭을 늘려 유의성을 탐색할 근거가 없다. 이번 표를 민감도 분석으로 보관하고, 기존에 논의한 업종 경기 자료를 통한 설명의 보완 여부를 판단하는 것이 다음 단계라는 제안이다. 어떤 업종을 추가로 삭제하거나 새로운 모형을 자동 실행하지 않는다.

## 검증·재현
원본 M1–M5 계수를 모두 재현했다. 제외 후 M2의 군집 공분산을 별도 행렬 공식으로 대조했다. 입력 해시, 행 중복, 모형 행렬 순위, 부트스트랩 고정효과 제거 후 계수 일치를 확인했다. 코드 3개 셀을 동일 Python 실행 공간에서 순서대로 실행하고 출력을 노트북에 저장했다. 제출용 노트북은 기존 최종 병합패널 이후의 분석을 재현하며, 최초 원자료에서 패널을 만드는 과정은 기존 01_analysis.ipynb에 있다.
